# nuPlan Mini Transfer Evaluation

This notebook turns the existing FM/CBF experiment into a nuPlan-mini transfer evaluation with:

- scenario filtering: `lane_following` and `left_turn`
- four methods: `FM+CBF`, `Pure FM`, `IDM`, `PDM-Closed`
- simplified metrics aligned with the 2D simulator
- ablations on `tau`, `K`, and road CBF margin

Implementation choice:

- ego is simulated in closed-loop with a kinematic bicycle + LQR tracker
- surrounding agents are replayed from nuPlan logs
- centerline is extracted from future expert ego poses, with a map-based hook left in place for later replacement
- the FM model is loaded from an existing checkpoint and is **not** trained or finetuned on nuPlan in this notebook

Recommended wording for paper/report use:

- `Pretrained transfer evaluation on nuPlan mini`
- `Zero-shot evaluation on nuPlan mini without nuPlan finetuning`

If the nuPlan archives are still zipped, set `CFG.extract_archives = True` before calling `prepare_nuplan_roots(CFG)`.


In [1]:
from __future__ import annotations

import math
import sys
import time
import warnings
import zipfile
from collections import defaultdict
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from scipy.ndimage import gaussian_filter1d
from scipy.spatial import KDTree

warnings.filterwarnings("ignore")
torch.set_grad_enabled(False)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from fmtorch.models.simple1d_unet import SimpleUNet1D
from dmpc_fm_cbf.sampler_5h import piecewise_sample_fm_5ch_with_cbf


@dataclass
class BenchmarkConfig:
    nuplan_mini_zip: Path = Path(r"C:\Users\ROG\Downloads\nuplan-v1.1_mini.zip")
    nuplan_maps_zip: Path = Path(r"C:\Users\ROG\Downloads\nuplan-maps-v1.0.zip")
    extract_root: Path = PROJECT_ROOT / "notebooks" / "_nuplan_cache"
    data_root_name: str = "nuplan-v1.1_mini"
    map_root_name: str = "nuplan-maps-v1.0"
    extract_archives: bool = False
    map_version: str = "nuplan-maps-v1.0"
    scenario_types: Tuple[str, ...] = ("lane_following", "left_turn")
    scenarios_per_type: int = 50
    shuffle_scenarios: bool = False
    random_seed: int = 7
    checkpoint_path: Path = PROJECT_ROOT / "notebooks" / "cache" / "model_vel_5ch_canonical_r.pt"
    output_root: Path = PROJECT_ROOT / "notebooks" / "nuplan_transfer_results"
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    dt: float = 0.1
    horizon_steps: int = 40
    max_episode_steps: int = 80
    replan_every: int = 2
    goal_tol: float = 2.0
    safe_radius: float = 2.5
    local_goal_distance: float = 12.0
    road_half_width: float = 4.0
    road_margin: float = 0.40
    road_cbf_alpha: float = 8.0
    road_boundary_points: int = 18
    road_boundary_radius: float = 1.0
    max_agents: int = 8
    fm_candidates: int = 6
    fm_tau0: float = 0.5
    fm_tau1: float = 0.9
    fm_num_segments: int = 5
    fm_noise_scale: float = 0.08
    fm_start_weight: float = 0.55
    fm_goal_weight: float = 0.95
    max_speed: float = 8.0
    accel_max: float = 3.0
    steer_max: float = math.radians(35.0)
    steer_rate_max: float = math.radians(90.0)
    wheel_base: float = 3.0
    cbf_agent_kwargs: Dict[str, float] = None
    cbf_road_kwargs: Dict[str, float] = None

    def __post_init__(self):
        if self.cbf_agent_kwargs is None:
            self.cbf_agent_kwargs = dict(passes=6, extra_margin=0.6, alpha=20.0, max_corr_norm=3.0)
        if self.cbf_road_kwargs is None:
            self.cbf_road_kwargs = dict(passes=3, extra_margin=0.1, alpha=12.0, max_corr_norm=2.0)

    @property
    def data_root(self) -> Path:
        return self.extract_root / self.data_root_name

    @property
    def map_root(self) -> Path:
        return self.extract_root / self.map_root_name


CFG = BenchmarkConfig()
CFG.output_root.mkdir(parents=True, exist_ok=True)


def clone_cfg(cfg: BenchmarkConfig, **updates: Any) -> BenchmarkConfig:
    data = asdict(cfg)
    data.update(updates)
    return BenchmarkConfig(**data)


def maybe_extract_archive(zip_path: Path, target_dir: Path, enabled: bool) -> Path:
    if target_dir.exists():
        return target_dir
    if not enabled:
        raise FileNotFoundError(
            f"Missing extracted dir: {target_dir}. "
            f"Either extract {zip_path.name} manually or set CFG.extract_archives=True."
        )
    if not zip_path.exists():
        raise FileNotFoundError(f"Archive not found: {zip_path}")
    target_dir.parent.mkdir(parents=True, exist_ok=True)
    print(f"Extracting {zip_path.name} -> {target_dir.parent} ...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(target_dir.parent)
    if not target_dir.exists():
        raise FileNotFoundError(f"Expected extracted dir not found after unzip: {target_dir}")
    return target_dir


def prepare_nuplan_roots(cfg: BenchmarkConfig) -> Tuple[Path, Path]:
    data_root = maybe_extract_archive(cfg.nuplan_mini_zip, cfg.data_root, cfg.extract_archives)
    map_root = maybe_extract_archive(cfg.nuplan_maps_zip, cfg.map_root, cfg.extract_archives)
    return data_root, map_root


def load_model(cfg: BenchmarkConfig) -> torch.nn.Module:
    if not cfg.checkpoint_path.exists():
        raise FileNotFoundError(f"Model checkpoint not found: {cfg.checkpoint_path}")
    model = SimpleUNet1D(
        time_emb_dim=128,
        hidden_dim=128,
        cond_dim=5,
        in_channels=5,
        out_channels=5,
    )
    ckpt = torch.load(cfg.checkpoint_path, map_location="cpu")
    state_dict = ckpt["model_state_dict"] if isinstance(ckpt, dict) and "model_state_dict" in ckpt else ckpt
    model.load_state_dict(state_dict, strict=True)
    return model.to(cfg.device).eval()


def _import_nuplan_api():
    try:
        from nuplan.planning.scenario_builder.nuplan_db.nuplan_scenario_builder import NuPlanScenarioBuilder
        from nuplan.planning.scenario_builder.scenario_filter import ScenarioFilter
    except Exception as exc:
        raise ImportError(
            "nuplan-devkit is required for this notebook. Install it in the current environment first."
        ) from exc
    return NuPlanScenarioBuilder, ScenarioFilter


def canonical_scenario_type(name: str) -> Optional[str]:
    key = (name or "").lower()
    if "lane_following" in key:
        return "lane_following"
    if "left_turn" in key:
        return "left_turn"
    return None


def load_scenarios(cfg: BenchmarkConfig):
    NuPlanScenarioBuilder, ScenarioFilter = _import_nuplan_api()
    data_root, map_root = prepare_nuplan_roots(cfg)
    db_files = str(data_root / "**" / "*.db")
    builder = NuPlanScenarioBuilder(
        data_root=str(data_root),
        map_root=str(map_root),
        sensor_root=None,
        db_files=db_files,
        map_version=cfg.map_version,
    )
    requested = list(cfg.scenario_types)
    raw_types = ["lane_following", "left_turn", "unprotected_left_turn"]
    scenario_filter = ScenarioFilter(
        scenario_types=raw_types,
        scenario_tokens=None,
        log_names=None,
        map_names=None,
        num_scenarios_per_type=cfg.scenarios_per_type,
        limit_total_scenarios=cfg.scenarios_per_type * max(2, len(raw_types)),
        shuffle=cfg.shuffle_scenarios,
        timestamp_threshold_s=None,
    )
    scenarios = builder.get_scenarios(scenario_filter, worker=None)
    grouped: Dict[str, List[Any]] = defaultdict(list)
    for scenario in scenarios:
        sc_type = canonical_scenario_type(getattr(scenario, "scenario_type", ""))
        if sc_type in requested and len(grouped[sc_type]) < cfg.scenarios_per_type:
            grouped[sc_type].append(scenario)
    selected = []
    for sc_type in requested:
        if len(grouped[sc_type]) == 0:
            print(f"[warn] no scenarios found for type={sc_type}")
        selected.extend(grouped[sc_type])
    if not selected:
        raise RuntimeError("No nuPlan scenarios matched the requested types.")
    print({k: len(v) for k, v in grouped.items()})
    return grouped, selected


def wrap_pi(angle: float) -> float:
    return float((angle + math.pi) % (2.0 * math.pi) - math.pi)


def ego_state_to_array(ego_state: Any) -> np.ndarray:
    rear_axle = ego_state.rear_axle
    dyn = ego_state.dynamic_car_state
    delta = 0.0
    if hasattr(ego_state, "tire_steering_angle"):
        delta = float(getattr(ego_state, "tire_steering_angle"))
    elif hasattr(dyn, "tire_steering_angle"):
        delta = float(getattr(dyn, "tire_steering_angle"))
    return np.array([
        float(rear_axle.x),
        float(rear_axle.y),
        float(rear_axle.heading),
        float(getattr(dyn, "speed", 0.0)),
        delta,
    ], dtype=np.float32)


def _tracked_objects_as_list(observation: Any) -> List[Any]:
    objs = observation
    if hasattr(observation, "tracked_objects"):
        objs = observation.tracked_objects
    if hasattr(objs, "get_agents"):
        try:
            return list(objs.get_agents())
        except Exception:
            pass
    if hasattr(objs, "tracked_objects"):
        try:
            return list(objs.tracked_objects)
        except Exception:
            pass
    return list(objs)


def tracked_object_to_state(obj: Any) -> Optional[np.ndarray]:
    try:
        center = obj.center
        vx = float(getattr(getattr(obj, "velocity", None), "x", 0.0))
        vy = float(getattr(getattr(obj, "velocity", None), "y", 0.0))
        speed = float(math.hypot(vx, vy))
        heading = float(getattr(center, "heading", 0.0))
        return np.array([float(center.x), float(center.y), heading, speed, 0.0], dtype=np.float32)
    except Exception:
        return None


def future_expert_centerline(scenario: Any, iteration: int, max_points: int = 80, step: int = 2) -> np.ndarray:
    n_total = int(scenario.get_number_of_iterations())
    idxs = list(range(iteration, min(n_total, iteration + max_points * step), step))
    pts = []
    for idx in idxs:
        ego = scenario.get_ego_state_at_iteration(idx)
        pts.append([float(ego.rear_axle.x), float(ego.rear_axle.y)])
    if len(pts) < 2:
        ego = scenario.get_ego_state_at_iteration(iteration)
        x0, y0 = float(ego.rear_axle.x), float(ego.rear_axle.y)
        hd = float(ego.rear_axle.heading)
        pts = [[x0 + 1.5 * k * math.cos(hd), y0 + 1.5 * k * math.sin(hd)] for k in range(2, 30)]
        pts.insert(0, [x0, y0])
    arr = np.asarray(pts, dtype=np.float32)
    if len(arr) > 4:
        arr[:, 0] = gaussian_filter1d(arr[:, 0].astype(float), sigma=1.2).astype(np.float32)
        arr[:, 1] = gaussian_filter1d(arr[:, 1].astype(float), sigma=1.2).astype(np.float32)
    return arr


def arclength(centerline: np.ndarray) -> np.ndarray:
    seg = np.linalg.norm(np.diff(centerline, axis=0), axis=1)
    return np.concatenate([[0.0], np.cumsum(seg)]).astype(np.float32)


def point_on_centerline(centerline: np.ndarray, distance_ahead: float) -> np.ndarray:
    arc = arclength(centerline)
    s = float(np.clip(distance_ahead, 0.0, arc[-1]))
    idx = int(np.searchsorted(arc, s, side="right") - 1)
    idx = int(np.clip(idx, 0, len(centerline) - 2))
    ds = float(max(arc[idx + 1] - arc[idx], 1e-6))
    ratio = float((s - arc[idx]) / ds)
    return centerline[idx] + ratio * (centerline[idx + 1] - centerline[idx])


def nearest_agents(ego_xy: np.ndarray, observation: Any, max_agents: int) -> np.ndarray:
    states = []
    for obj in _tracked_objects_as_list(observation):
        st = tracked_object_to_state(obj)
        if st is None:
            continue
        states.append(st)
    if not states:
        return np.zeros((0, 5), dtype=np.float32)
    arr = np.asarray(states, dtype=np.float32)
    d = np.linalg.norm(arr[:, :2] - ego_xy.reshape(1, 2), axis=1)
    order = np.argsort(d)[:max_agents]
    return arr[order]


def extract_frame(scenario: Any, iteration: int, cfg: BenchmarkConfig, ego_override: Optional[np.ndarray] = None) -> Dict[str, Any]:
    ego_logged = scenario.get_ego_state_at_iteration(iteration)
    ego_state = ego_state_to_array(ego_logged) if ego_override is None else ego_override.astype(np.float32).copy()
    observation = scenario.get_tracked_objects_at_iteration(iteration)
    centerline = future_expert_centerline(scenario, iteration)
    local_goal_xy = point_on_centerline(centerline, cfg.local_goal_distance)
    mission_goal = scenario.get_mission_goal()
    if mission_goal is not None:
        mission_goal_xy = np.array([float(mission_goal.x), float(mission_goal.y)], dtype=np.float32)
    else:
        mission_goal_xy = centerline[-1].astype(np.float32)
    agents = nearest_agents(ego_state[:2], observation, cfg.max_agents)
    return {
        "scenario_token": getattr(scenario, "token", f"iter_{iteration}"),
        "scenario_type": canonical_scenario_type(getattr(scenario, "scenario_type", "")) or getattr(scenario, "scenario_type", "unknown"),
        "ego_state": ego_state,
        "agents": agents,
        "goal_xy": local_goal_xy.astype(np.float32),
        "mission_goal_xy": mission_goal_xy,
        "centerline": centerline,
        "road_half_width": float(cfg.road_half_width),
    }


def bicycle_step(state: np.ndarray, control: np.ndarray, cfg: BenchmarkConfig) -> np.ndarray:
    x, y, heading, speed, delta = state
    a = float(np.clip(control[0], -cfg.accel_max, cfg.accel_max))
    delta_rate = float(np.clip(control[1], -cfg.steer_rate_max, cfg.steer_rate_max))
    speed2 = float(np.clip(speed + cfg.dt * a, 0.0, cfg.max_speed))
    delta2 = float(np.clip(delta + cfg.dt * delta_rate, -cfg.steer_max, cfg.steer_max))
    heading2 = wrap_pi(heading + cfg.dt * (speed2 / cfg.wheel_base) * math.tan(delta2))
    x2 = x + cfg.dt * speed2 * math.cos(heading2)
    y2 = y + cfg.dt * speed2 * math.sin(heading2)
    return np.array([x2, y2, heading2, speed2, delta2], dtype=np.float32)


_Q = np.diag([8.0, 10.0, 1.0]).astype(np.float64)
_R = np.diag([1.0, 1.0]).astype(np.float64)


def _dare(A: np.ndarray, B: np.ndarray, Q: np.ndarray, R: np.ndarray, n: int = 200) -> np.ndarray:
    P = Q.copy()
    for _ in range(n):
        Pn = A.T @ P @ A - A.T @ P @ B @ np.linalg.inv(R + B.T @ P @ B) @ B.T @ P @ A + Q
        if np.max(np.abs(Pn - P)) < 1e-8:
            break
        P = Pn
    return P


def _lqr_gain(v_ref: float, cfg: BenchmarkConfig) -> np.ndarray:
    v = max(v_ref, 0.1)
    A_c = np.array([[0.0, v, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]], dtype=np.float64)
    B_c = np.array([[0.0, 0.0], [0.0, v / cfg.wheel_base], [1.0, 0.0]], dtype=np.float64)
    A_d = np.eye(3) + cfg.dt * A_c
    B_d = cfg.dt * B_c
    P = _dare(A_d, B_d, _Q, _R)
    return np.linalg.inv(_R + B_d.T @ P @ B_d) @ B_d.T @ P @ A_d


def lqr_control(state: np.ndarray, ref_path: np.ndarray, goal_xy: np.ndarray, cfg: BenchmarkConfig) -> np.ndarray:
    pos = state[:2].astype(np.float64)
    heading = float(state[2])
    speed = float(state[3])
    delta = float(state[4])
    d_goal = float(np.linalg.norm(pos - goal_xy.astype(np.float64)))
    lookahead = float(np.clip(4.0 + 0.5 * speed, 2.0, 8.0))
    dist = np.linalg.norm(ref_path - pos.reshape(1, 2), axis=1)
    near = int(np.argmin(dist))
    target = ref_path[-1]
    for i in range(near, len(ref_path)):
        if np.linalg.norm(ref_path[i] - pos) >= lookahead:
            target = ref_path[i]
            break
    if d_goal < 2.5:
        v_ref = 0.0
    elif d_goal < 8.0:
        v_ref = cfg.max_speed * 0.5 * d_goal / 8.0
    else:
        v_ref = cfg.max_speed
    dx = float(target[0] - pos[0])
    dy = float(target[1] - pos[1])
    heading_ref = math.atan2(dy, dx)
    heading_err = wrap_pi(heading_ref - heading)
    if len(ref_path) > 1:
        seg = ref_path[min(near + 1, len(ref_path) - 1)] - ref_path[near]
        seg_len = float(np.linalg.norm(seg) + 1e-9)
        n_hat = np.array([-seg[1], seg[0]], dtype=np.float64) / seg_len
        lateral_err = float(np.dot(pos - ref_path[near], n_hat))
    else:
        lateral_err = 0.0
    err = np.array([lateral_err, heading_err, v_ref - speed], dtype=np.float64)
    K = _lqr_gain(max(speed, 0.1), cfg)
    u = K @ err
    dist_t = max(1e-3, float(np.linalg.norm(target - pos)))
    kappa = 2.0 * math.sin(heading_err) / dist_t
    delta_ref = float(np.clip(math.atan(cfg.wheel_base * kappa), -cfg.steer_max, cfg.steer_max))
    delta_rate_ref = float(np.clip((delta_ref - delta) / cfg.dt, -cfg.steer_rate_max, cfg.steer_rate_max))
    return np.array([
        float(np.clip(u[0], -cfg.accel_max, cfg.accel_max)),
        float(np.clip(u[1] + 0.5 * delta_rate_ref, -cfg.steer_rate_max, cfg.steer_rate_max)),
    ], dtype=np.float32)


def _get_lateral(pos_xy: np.ndarray, centerline: np.ndarray) -> Tuple[float, np.ndarray, np.ndarray, np.ndarray]:
    kdt = KDTree(centerline)
    _, idx = kdt.query(pos_xy)
    i0 = max(int(idx) - 1, 0)
    i1 = min(int(idx) + 1, len(centerline) - 1)
    tangent = centerline[i1] - centerline[i0]
    tangent /= float(np.linalg.norm(tangent) + 1e-9)
    normal = np.array([-tangent[1], tangent[0]], dtype=np.float32)
    d_lat = float(np.dot(pos_xy - centerline[int(idx)], normal))
    return d_lat, tangent.astype(np.float32), normal, centerline[int(idx)]


def reparametrize_to_centerline(xy: np.ndarray, centerline: np.ndarray, road_half_width: float) -> np.ndarray:
    if len(xy) < 2:
        return xy.astype(np.float32)
    arc = arclength(centerline)
    kdt = KDTree(centerline)
    s_vals = np.zeros(len(xy), dtype=np.float32)
    lat_vals = np.zeros(len(xy), dtype=np.float32)
    for k, point in enumerate(xy):
        _, idx = kdt.query(point)
        i0 = max(int(idx) - 1, 0)
        i1 = min(int(idx) + 1, len(centerline) - 1)
        tangent = centerline[i1] - centerline[i0]
        tangent /= float(np.linalg.norm(tangent) + 1e-9)
        normal = np.array([-tangent[1], tangent[0]], dtype=np.float32)
        rel = point - centerline[int(idx)]
        s_vals[k] = float(arc[int(idx)] + np.dot(rel, tangent))
        lat_vals[k] = float(np.dot(rel, normal))
    for k in range(1, len(s_vals)):
        s_vals[k] = max(s_vals[k], s_vals[k - 1] + 1e-3)
    lat_vals = gaussian_filter1d(lat_vals.astype(float), sigma=1.2)
    lat_vals = np.clip(lat_vals, -road_half_width, road_half_width)
    out = np.zeros_like(xy, dtype=np.float32)
    for k, s in enumerate(s_vals):
        s = float(np.clip(s, 0.0, arc[-1]))
        idx = int(np.searchsorted(arc, s, side="right") - 1)
        idx = int(np.clip(idx, 0, len(centerline) - 2))
        ds = float(max(arc[idx + 1] - arc[idx], 1e-6))
        ratio = (s - arc[idx]) / ds
        base = centerline[idx] + ratio * (centerline[idx + 1] - centerline[idx])
        tangent = centerline[min(idx + 1, len(centerline) - 1)] - centerline[max(idx - 1, 0)]
        tangent /= float(np.linalg.norm(tangent) + 1e-9)
        normal = np.array([-tangent[1], tangent[0]], dtype=np.float32)
        out[k] = base + float(lat_vals[k]) * normal
    return out.astype(np.float32)


def build_canonical_tf(start_xy: np.ndarray, goal_xy: np.ndarray) -> Dict[str, Any]:
    start = np.asarray(start_xy, dtype=np.float32).reshape(2)
    goal = np.asarray(goal_xy, dtype=np.float32).reshape(2)
    diff = goal - start
    dist = float(np.linalg.norm(diff) + 1e-9)
    c = float(diff[0] / dist)
    s = float(diff[1] / dist)
    R_wc = np.array([[c, s], [-s, c]], dtype=np.float32)
    R_cw = R_wc.T
    phi = math.atan2(diff[1], diff[0])
    def w2c(xy):
        arr = np.asarray(xy, dtype=np.float32).reshape(-1, 2)
        return (arr - start) @ R_wc.T
    def c2w(xy):
        arr = np.asarray(xy, dtype=np.float32).reshape(-1, 2)
        return arr @ R_cw.T + start
    return {
        "w2c": w2c,
        "c2w": c2w,
        "w2c_angle": lambda th: wrap_pi(float(th) - phi),
        "start_can": np.array([0.0, 0.0], dtype=np.float32),
        "goal_can": np.array([dist, 0.0], dtype=np.float32),
        "dist": dist,
    }


def build_agent_ellipses(frame: Dict[str, Any], tf: Dict[str, Any], cfg: BenchmarkConfig) -> List[Dict[str, Any]]:
    ellipses: List[Dict[str, Any]] = []
    for agent in frame["agents"]:
        agent_can = tf["w2c"](agent[:2].reshape(1, 2))[0]
        ellipses.append({"center": agent_can.copy(), "radius": float(cfg.safe_radius)})
    return ellipses


def build_road_boundary_ellipses(frame: Dict[str, Any], tf: Dict[str, Any], cfg: BenchmarkConfig) -> List[Dict[str, Any]]:
    centerline = frame["centerline"]
    pos_xy = frame["ego_state"][:2]
    kdt = KDTree(centerline)
    _, near = kdt.query(pos_xy)
    horizon_idx = np.linspace(int(near), len(centerline) - 1, cfg.road_boundary_points).astype(int)
    ellipses = []
    for idx in horizon_idx:
        i0 = max(int(idx) - 1, 0)
        i1 = min(int(idx) + 1, len(centerline) - 1)
        tangent = centerline[i1] - centerline[i0]
        tangent /= float(np.linalg.norm(tangent) + 1e-9)
        normal = np.array([-tangent[1], tangent[0]], dtype=np.float32)
        for side in (+1.0, -1.0):
            bound_point = centerline[int(idx)] + side * (cfg.road_half_width - cfg.road_margin) * normal
            ellipses.append({
                "center": tf["w2c"](bound_point.reshape(1, 2))[0].copy(),
                "radius": float(cfg.road_boundary_radius),
            })
    return ellipses


def cbf_road_control_filter(control: np.ndarray, state: np.ndarray, centerline: np.ndarray, cfg: BenchmarkConfig) -> Tuple[np.ndarray, float]:
    d_lat, _, normal, _ = _get_lateral(state[:2], centerline)
    h = float(cfg.road_half_width - cfg.road_margin - abs(d_lat))
    if h > 1.0:
        return control.copy(), h
    speed = float(state[3])
    vel = np.array([speed * math.cos(float(state[2])), speed * math.sin(float(state[2]))], dtype=np.float32)
    sign = 1.0 if d_lat >= 0.0 else -1.0
    h_dot = -sign * float(np.dot(vel, normal))
    slack = h_dot + cfg.road_cbf_alpha * h
    if slack >= 0.0:
        return control.copy(), h
    sensitivity = max(0.1, abs(speed) * cfg.dt * (abs(speed) / cfg.wheel_base + 0.2))
    correction = -sign * (-slack / sensitivity)
    safe = control.copy()
    safe[1] = float(np.clip(safe[1] + correction, -cfg.steer_rate_max, cfg.steer_rate_max))
    return safe.astype(np.float32), h


def fm_sample_path(model: torch.nn.Module, state: np.ndarray, frame: Dict[str, Any], cfg: BenchmarkConfig, *, use_agent_cbf: bool, use_road_cbf: bool, seed: int) -> np.ndarray:
    tf = build_canonical_tf(state[:2], frame["goal_xy"])
    heading_can = tf["w2c_angle"](float(state[2]))
    cond = np.array([
        float(tf["dist"]),
        float(state[3]),
        math.cos(heading_can),
        math.sin(heading_can),
        float(state[4]),
    ], dtype=np.float32)
    ellipses: List[Dict[str, Any]] = []
    cbf_kwargs = None
    if use_agent_cbf:
        ellipses.extend(build_agent_ellipses(frame, tf, cfg))
        cbf_kwargs = dict(cfg.cbf_agent_kwargs)
    if use_road_cbf:
        ellipses.extend(build_road_boundary_ellipses(frame, tf, cfg))
        if cbf_kwargs is None:
            cbf_kwargs = dict(cfg.cbf_road_kwargs)
    generator = torch.Generator(device=cfg.device).manual_seed(int(seed))
    x0 = torch.randn((1, 5, cfg.horizon_steps), generator=generator, device=cfg.device) * float(cfg.fm_noise_scale)
    x0[:, :, 0] = 0.0
    x_fm, _, _ = piecewise_sample_fm_5ch_with_cbf(
        model_5ch=model,
        T_steps=cfg.horizon_steps,
        device=cfg.device,
        total_time=1.0,
        num_segments=cfg.fm_num_segments,
        ode_method="euler",
        atol=1e-4,
        rtol=1e-4,
        noise_scale=cfg.fm_noise_scale,
        x0_noise=x0,
        cond=cond,
        start_xy_norm=tf["start_can"],
        goal_xy_norm=tf["goal_can"],
        start_guidance_weight=cfg.fm_start_weight,
        goal_guidance_weight=cfg.fm_goal_weight,
        goal_guidance_mode="tail",
        lock_start=True,
        use_cbf=bool(ellipses),
        scaler=None,
        ellipses=ellipses,
        tau0=cfg.fm_tau0,
        tau1=cfg.fm_tau1,
        cbf_kwargs=cbf_kwargs,
    )
    xy_can = x_fm[0:2, :].T.astype(np.float32)
    xy_world = tf["c2w"](xy_can).astype(np.float32)
    return reparametrize_to_centerline(xy_world, frame["centerline"], cfg.road_half_width)


def fm_plan(model: torch.nn.Module, state: np.ndarray, frame: Dict[str, Any], cfg: BenchmarkConfig, *, use_agent_cbf: bool, use_road_cbf: bool, seed: int) -> np.ndarray:
    best_path = None
    best_cost = float("inf")
    for k in range(cfg.fm_candidates):
        path = fm_sample_path(
            model,
            state,
            frame,
            cfg,
            use_agent_cbf=use_agent_cbf,
            use_road_cbf=use_road_cbf,
            seed=seed + 7919 * (k + 1),
        )
        d_end = float(np.linalg.norm(path[-1] - frame["goal_xy"]))
        d_mid = float(np.mean(np.linalg.norm(path - frame["goal_xy"].reshape(1, 2), axis=1)))
        coll_penalty = 0.0
        if len(frame["agents"]) > 0:
            d = np.linalg.norm(path[:, None, :] - frame["agents"][None, :, :2], axis=2).min()
            coll_penalty = 200.0 * max(0.0, cfg.safe_radius - float(d)) ** 2
        road_penalty = float(np.mean(np.abs([_get_lateral(p, frame["centerline"])[0] for p in path])))
        cost = 18.0 * d_end + 1.5 * d_mid + coll_penalty + 0.3 * road_penalty
        if cost < best_cost:
            best_cost = cost
            best_path = path
    return best_path if best_path is not None else frame["centerline"][: cfg.horizon_steps].copy()


def idm_plan(state: np.ndarray, frame: Dict[str, Any], cfg: BenchmarkConfig) -> np.ndarray:
    path = []
    pos = state[:2].copy()
    speed = float(state[3])
    centerline = frame["centerline"]
    kdt = KDTree(centerline)
    for _ in range(cfg.horizon_steps):
        _, idx = kdt.query(pos)
        idx = min(int(idx) + 1, len(centerline) - 1)
        target = centerline[idx]
        direction = target - pos
        norm = float(np.linalg.norm(direction) + 1e-9)
        direction = direction / norm
        gap = float("inf")
        if len(frame["agents"]) > 0:
            gap = float(np.linalg.norm(frame["agents"][:, :2] - pos.reshape(1, 2), axis=1).min())
        desired = cfg.max_speed
        if gap < 15.0:
            desired = cfg.max_speed * max(0.1, (gap - cfg.safe_radius) / max(15.0 - cfg.safe_radius, 1e-6))
        speed = float(np.clip(speed + cfg.dt * 1.5 * (desired - speed), 0.0, cfg.max_speed))
        pos = pos + cfg.dt * speed * direction.astype(np.float32)
        path.append(pos.copy())
    return reparametrize_to_centerline(np.asarray(path, dtype=np.float32), centerline, cfg.road_half_width)


def pdm_closed_plan(state: np.ndarray, frame: Dict[str, Any], cfg: BenchmarkConfig) -> np.ndarray:
    path = []
    pos = state[:2].copy()
    centerline = frame["centerline"]
    arc = arclength(centerline)
    kdt = KDTree(centerline)
    for _ in range(cfg.horizon_steps):
        _, idx = kdt.query(pos)
        s_now = float(arc[int(idx)])
        target = point_on_centerline(centerline, s_now + 3.0)
        direction = target - pos
        direction /= float(np.linalg.norm(direction) + 1e-9)
        speed = cfg.max_speed * 0.8
        if len(frame["agents"]) > 0:
            d_obs = float(np.linalg.norm(frame["agents"][:, :2] - pos.reshape(1, 2), axis=1).min())
            if d_obs < 12.0:
                speed *= max(0.15, (d_obs - cfg.safe_radius) / max(12.0 - cfg.safe_radius, 1e-6))
        pos = pos + cfg.dt * speed * direction.astype(np.float32)
        path.append(pos.copy())
    return reparametrize_to_centerline(np.asarray(path, dtype=np.float32), centerline, cfg.road_half_width)


METHODS = {
    "fm_cbf": "FM+CBF",
    "pure_fm": "Pure FM",
    "idm": "IDM",
    "pdm_closed": "PDM-Closed",
}


def plan_path(method: str, model: torch.nn.Module, state: np.ndarray, frame: Dict[str, Any], cfg: BenchmarkConfig, seed: int) -> np.ndarray:
    if method == "fm_cbf":
        return fm_plan(model, state, frame, cfg, use_agent_cbf=True, use_road_cbf=True, seed=seed)
    if method == "pure_fm":
        return fm_plan(model, state, frame, cfg, use_agent_cbf=False, use_road_cbf=False, seed=seed)
    if method == "idm":
        return idm_plan(state, frame, cfg)
    if method == "pdm_closed":
        return pdm_closed_plan(state, frame, cfg)
    raise ValueError(f"Unknown method: {method}")


def jerk_rms(accels: Sequence[float], dt: float) -> float:
    arr = np.asarray(list(accels), dtype=np.float32)
    if len(arr) < 2:
        return 0.0
    j = np.diff(arr) / max(dt, 1e-6)
    return float(np.sqrt(np.mean(j ** 2)))


def run_episode(scenario: Any, method: str, model: torch.nn.Module, cfg: BenchmarkConfig, seed: int = 0) -> Dict[str, Any]:
    n_iters = min(int(scenario.get_number_of_iterations()), cfg.max_episode_steps)
    frame0 = extract_frame(scenario, 0, cfg)
    state = frame0["ego_state"].copy()
    traj = [state.copy()]
    accels: List[float] = []
    min_agent_dist = float("inf")
    road_violations = 0
    collision = False
    inference_times: List[float] = []
    road_h_log: List[float] = []
    goal_reached = False
    mission_goal_xy = frame0["mission_goal_xy"]
    active_path = frame0["centerline"][: cfg.horizon_steps].copy()
    dt_local = float(getattr(scenario, "database_interval", cfg.dt))
    local_cfg = clone_cfg(cfg, dt=dt_local)
    for step in range(n_iters):
        frame = extract_frame(scenario, step, local_cfg, ego_override=state)
        frame["mission_goal_xy"] = mission_goal_xy
        if len(frame["agents"]) > 0:
            dist_agents = np.linalg.norm(frame["agents"][:, :2] - state[:2].reshape(1, 2), axis=1)
            min_agent_dist = min(min_agent_dist, float(dist_agents.min()))
            if float(dist_agents.min()) < local_cfg.safe_radius:
                collision = True
        h_now = float(local_cfg.road_half_width - local_cfg.road_margin - abs(_get_lateral(state[:2], frame["centerline"])[0]))
        road_h_log.append(h_now)
        if h_now < 0.0:
            road_violations += 1
        if float(np.linalg.norm(state[:2] - mission_goal_xy)) < local_cfg.goal_tol:
            goal_reached = True
            break
        if step % local_cfg.replan_every == 0:
            t0 = time.time()
            active_path = plan_path(method, model, state, frame, local_cfg, seed=seed + 97 * step)
            inference_times.append((time.time() - t0) * 1000.0)
        control = lqr_control(state, active_path, frame["goal_xy"], local_cfg)
        if method == "fm_cbf":
            control, _ = cbf_road_control_filter(control, state, frame["centerline"], local_cfg)
        accels.append(float(control[0]))
        state = bicycle_step(state, control, local_cfg)
        traj.append(state.copy())
    traj_arr = np.asarray(traj, dtype=np.float32)
    if not np.isfinite(min_agent_dist):
        min_agent_dist = float("nan")
    return {
        "scenario_token": getattr(scenario, "token", "unknown"),
        "scenario_type": canonical_scenario_type(getattr(scenario, "scenario_type", "")) or getattr(scenario, "scenario_type", "unknown"),
        "method": METHODS[method],
        "method_key": method,
        "steps": int(len(traj_arr) - 1),
        "time_to_goal": float((len(traj_arr) - 1) * local_cfg.dt),
        "collision": bool(collision),
        "goal_reached": bool(goal_reached),
        "dist_to_goal": float(np.linalg.norm(traj_arr[-1, :2] - mission_goal_xy)),
        "min_inter_agent_dist": float(min_agent_dist),
        "road_boundary_violations": int(road_violations),
        "mean_inference_time_ms": float(np.mean(inference_times)) if inference_times else 0.0,
        "jerk_rms": jerk_rms(accels, local_cfg.dt),
        "road_h_min": float(np.min(road_h_log)) if road_h_log else float("nan"),
        "trajectory": traj_arr,
        "mission_goal_xy": mission_goal_xy,
    }


def run_suite(scenarios: Sequence[Any], model: torch.nn.Module, cfg: BenchmarkConfig, methods: Optional[Sequence[str]] = None) -> pd.DataFrame:
    methods = list(methods or METHODS.keys())
    rows = []
    for sc_idx, scenario in enumerate(scenarios):
        for method_idx, method in enumerate(methods):
            seed = cfg.random_seed + 1000 * sc_idx + 17 * method_idx
            rows.append(run_episode(scenario, method, model, cfg, seed=seed))
    return pd.DataFrame(rows)


def compute_metrics(episode_df: pd.DataFrame) -> pd.DataFrame:
    cols = [
        "time_to_goal",
        "collision",
        "goal_reached",
        "dist_to_goal",
        "min_inter_agent_dist",
        "road_boundary_violations",
        "mean_inference_time_ms",
        "jerk_rms",
        "road_h_min",
    ]
    meta = episode_df[["scenario_type", "method"] + cols].copy()
    summary = meta.groupby(["scenario_type", "method"], as_index=False).agg(
        episodes=("method", "size"),
        time_to_goal=("time_to_goal", "mean"),
        collision_rate=("collision", "mean"),
        goal_reached_rate=("goal_reached", "mean"),
        dist_to_goal=("dist_to_goal", "mean"),
        min_inter_agent_dist=("min_inter_agent_dist", "mean"),
        road_boundary_violations=("road_boundary_violations", "mean"),
        mean_inference_time_ms=("mean_inference_time_ms", "mean"),
        jerk_rms=("jerk_rms", "mean"),
        road_h_min=("road_h_min", "mean"),
    )
    return summary


def plot_results(summary_df: pd.DataFrame, output_dir: Path, prefix: str = "transfer_eval") -> None:
    output_dir.mkdir(parents=True, exist_ok=True)
    metric_specs = [
        ("time_to_goal", "Time to Goal", True),
        ("collision_rate", "Collision Rate", True),
        ("goal_reached_rate", "Goal Reached Rate", False),
        ("min_inter_agent_dist", "Min Inter-Agent Dist", False),
        ("road_boundary_violations", "Road Boundary Violations", True),
        ("mean_inference_time_ms", "Inference Time (ms)", True),
    ]
    scenario_types = list(summary_df["scenario_type"].drop_duplicates())
    methods = list(summary_df["method"].drop_duplicates())
    fig, axes = plt.subplots(2, 3, figsize=(17, 8), constrained_layout=True)
    for ax, (key, title, lower_is_better) in zip(axes.ravel(), metric_specs):
        width = 0.8 / max(1, len(methods))
        x = np.arange(len(scenario_types))
        for i, method in enumerate(methods):
            vals = []
            for sc in scenario_types:
                sub = summary_df[(summary_df["scenario_type"] == sc) & (summary_df["method"] == method)]
                vals.append(float(sub.iloc[0][key]) if len(sub) else np.nan)
            ax.bar(x + (i - 0.5 * (len(methods) - 1)) * width, vals, width=width, label=method)
        ax.set_title(title + (" ↓" if lower_is_better else " ↑"))
        ax.set_xticks(x)
        ax.set_xticklabels(scenario_types)
        ax.grid(True, axis="y", alpha=0.25)
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=len(methods))
    fig.suptitle("nuPlan Mini Transfer Evaluation (Pretrained Model, No nuPlan Finetuning)", fontsize=14)
    save_path = output_dir / f"{prefix}_metrics.png"
    fig.savefig(save_path, dpi=180, bbox_inches="tight")
    plt.show()
    print(f"Saved: {save_path}")


def save_results(episode_df: pd.DataFrame, summary_df: pd.DataFrame, output_dir: Path, prefix: str) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)
    ep = episode_df.drop(columns=["trajectory", "mission_goal_xy"], errors="ignore")
    ep.to_csv(output_dir / f"{prefix}_episodes.csv", index=False)
    summary_df.to_csv(output_dir / f"{prefix}_summary.csv", index=False)


def run_single_factor_ablation(scenarios: Sequence[Any], model: torch.nn.Module, cfg: BenchmarkConfig, factor_name: str, values: Sequence[Any], method: str = "fm_cbf") -> pd.DataFrame:
    rows = []
    for value in values:
        if factor_name == "tau0":
            cfg_i = clone_cfg(cfg, fm_tau0=float(value), fm_tau1=max(float(value) + 0.2, cfg.fm_tau1))
        elif factor_name == "k_candidates":
            cfg_i = clone_cfg(cfg, fm_candidates=int(value))
        elif factor_name == "road_margin":
            cfg_i = clone_cfg(cfg, road_margin=float(value))
        else:
            raise ValueError(f"Unsupported ablation factor: {factor_name}")
        df_i = run_suite(scenarios, model, cfg_i, methods=[method])
        df_i[factor_name] = value
        rows.append(df_i)
    return pd.concat(rows, ignore_index=True)


print(f"Project root : {PROJECT_ROOT}")
print(f"Output root  : {CFG.output_root}")
print(f"Device       : {CFG.device}")


Project root : E:\projects\lab2025\dmpc_fm_cbf
Output root  : E:\projects\lab2025\dmpc_fm_cbf\notebooks\nuplan_transfer_results
Device       : cuda


In [3]:
# Pretrained transfer evaluation on nuPlan mini
# No training or finetuning is performed in this notebook.
# For a smoke test, change CFG to clone_cfg(CFG, scenarios_per_type=2, max_episode_steps=20, fm_candidates=3)

cfg = CFG
model = load_model(cfg)
grouped_scenarios, selected_scenarios = load_scenarios(cfg)

episode_df = run_suite(selected_scenarios, model, cfg)
summary_df = compute_metrics(episode_df)
save_results(episode_df, summary_df, cfg.output_root, prefix="transfer_eval")
plot_results(summary_df, cfg.output_root, prefix="transfer_eval")
summary_df

# Ablation examples
# tau_df = run_single_factor_ablation(selected_scenarios, model, cfg, factor_name="tau0", values=[0.3, 0.5, 0.7])
# tau_summary = tau_df.groupby(["scenario_type", "tau0"], as_index=False).agg(collision_rate=("collision", "mean"), jerk_rms=("jerk_rms", "mean"))
# tau_summary

# k_df = run_single_factor_ablation(selected_scenarios, model, cfg, factor_name="k_candidates", values=[1, 3, 6, 9])
# k_summary = k_df.groupby(["scenario_type", "k_candidates"], as_index=False).agg(goal_reached_rate=("goal_reached", "mean"), mean_inference_time_ms=("mean_inference_time_ms", "mean"))
# k_summary

# margin_df = run_single_factor_ablation(selected_scenarios, model, cfg, factor_name="road_margin", values=[0.0, 0.4, 0.8])
# margin_summary = margin_df.groupby(["scenario_type", "road_margin"], as_index=False).agg(road_boundary_violations=("road_boundary_violations", "mean"), jerk_rms=("jerk_rms", "mean"))
# margin_summary


ImportError: nuplan-devkit is required for this notebook. Install it in the current environment first.

In [9]:
from nuplan.planning.scenario_builder.nuplan_db.nuplan_scenario_builder import NuPlanScenarioBuilder

ModuleNotFoundError: No module named 'geopandas'